In [ ]:
#IMPORT STATEMENTS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os


df = pd.read_csv("HI-Small_Trans-csv.csv") 





df.drop_duplicates(inplace=True)



df["Is Laundering"].value_counts(normalize=True)*100


# Convert the "Timestamp" column to datetime format
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# Extract Date, Day, and Time from the Timestamp
df["Date"] = df["Timestamp"].dt.date
df["Day"] = df["Timestamp"].dt.day_name()
df["Time"] = df["Timestamp"].dt.time                                              

df.drop(columns=["Timestamp"], inplace=True)




X = df.drop(['Is Laundering'],axis = 1)
y = df['Is Laundering']


# Define the undersampler
from imblearn.under_sampling import RandomUnderSampler
undersampler = RandomUnderSampler()



X,y=undersampler.fit_resample(X,y)



from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)


categorical = ['From Bank','Account','To Bank','Receiving Currency','Payment Currency','Payment Format','Day']


from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
ohe = Pipeline([('Encoder',OneHotEncoder(drop = 'first',handle_unknown='ignore'))])


from sklearn.compose import ColumnTransformer
transformer = ColumnTransformer([('One Hot Encoding',ohe,categorical)])


from xgboost import XGBClassifier
# Define the full pipeline without the undersampler
model = Pipeline([
    ('Transformer', transformer),           # Preprocess the data
    ('Estimator', XGBClassifier())          # Train the model with XGBoost
])




model.fit(X_train,y_train)

# Ensure the src folder exists first
os.makedirs('src', exist_ok=True)

joblib.dump(model, 'src/model.joblib')
joblib.dump(transformer, 'src/transformer.joblib')



['transformer.joblib']